In [ ]:
%load_ext manim

# Simulación de la suma de dados y aproximación a la distribución normal

Este proyecto utiliza **Manim** (v0.20.1) para visualizar cómo evoluciona la distribución de la suma de varios dados al repetir un experimento aleatorio muchas veces.

## Objetivos

- Simular lanzamientos de varios dados.
- Construir un histograma de frecuencias de forma incremental.
- Visualizar la distribución de probabilidad de un dado.
- Comparar los resultados experimentales con la distribución normal teórica.
- Ilustrar el Teorema Central del Límite de manera intuitiva.


In [ ]:
from manim import *
import numpy as np
import random
from math import comb
from scipy.stats import norm
from collections import defaultdict

## Parámetros de la simulación

En esta sección se definen los parámetros principales:

- Número de dados lanzados en cada experimento.
- Número total de experimentos.
- Velocidad de la animación.
- Altura objetivo del histograma.

Permite modificar elementro basicos sin tener que estar buscando en el código.

In [ ]:
# Número de dados
n_dados = 10

# Número de experimentos 
n_experimentos = 800

# Cada cuántos experimentos renderizar un frame (800 experimentos y salto de 4 → 200 frames animados)
salto_animacion = 4

# Duración de cada frame animado (en segundos)
run_time_animacion = 0.07

# Altura objetivo hasta la cual llega el histograma para la animación (en unidades de Manim)
altura_objetivo = 3

## Construcción visual de la clase `Dado`

Esta clase genera una representación visual de un dado.

Características:
- Bordes redondeados.
- Distribución correcta de puntos para cada cara.
- Tamaño configurable.

Cada instancia representa un dado mostrando un valor entre 1 y 6.

In [ ]:
# Clase para representar un dado
class Dado(VGroup):
    def __init__(self, valor=1, size=0.42):
        super().__init__()
        # Crear el borde del dado como un rectángulo redondeado con las especificaciones dadas
        borde = RoundedRectangle(
            corner_radius=0.08,
            width=size,
            height=size,
            stroke_width=1.8,
            stroke_color=WHITE,
            fill_color="#1f1f1f",
            fill_opacity=1
        )
         
        # Distancia desde el centro del dado para colocar los puntos (ajustada para que se vea bien)
        s = size * 0.24
        # Define las posiciones de los puntos para cada valor del dado
        posiciones = {
            1: [(0, 0)],
            2: [(-s, -s), (s, s)], 
            3: [(-s, -s), (0, 0), (s, s)],
            4: [(-s, -s), (-s, s), (s, -s), (s, s)],
            5: [(-s, -s), (-s, s), (0, 0), (s, -s), (s, s)],
            6: [(-s, -s), (-s, 0), (-s, s), (s, -s), (s, 0), (s, s)]
        }
        
        # Crea los puntos del dado según el valor dado
        puntos = VGroup(*[
            Dot(
                point=[x, y, 0], # Posición del punto en el dado
                radius=size * 0.06, # Tamaño del punto
                color="#5dade2" #Color
            )
            for x, y in posiciones[valor]
        ])
        
        # Agrega el cuadrado del dado y los puntos al grupo del dado
        self.add(borde, puntos)

## Construcció visual de la clase `Histograma`

Esta clase representa gráficamente la distribución de frecuencias de las sumas obtenidas al lanzar varios dados.
Cada posible suma se asocia a una columna del histograma. La altura de cada columna depende del número de veces que dicha suma ha aparecido durante la simulación.

### Estructura

El histograma está formado por:

- Un eje horizontal que contiene todas las sumas posibles.
- Una columna para cada suma.
- Etiquetas numéricas que identifican cada resultado (de 2 en 2).

### Actualización de frecuencias

Cuando se obtiene una nueva suma:

1. Se identifica la columna correspondiente a ese valor.
2. Se incrementa su frecuencia en una unidad.
3. La columna aumenta su altura para reflejar la nueva frecuencia observada.

De esta forma, el histograma evoluciona progresivamente conforme se realizan más experimentos hasta llegar a ajustarse a la curva de la normal.

### Interpretación

Las columnas más altas indican resultados más frecuentes, mientras que las más bajas representan resultados menos probables.
A medida que aumenta el número de lanzamientos, la forma del histograma se estabiliza y permite visualizar la distribución de probabilidad asociada a la suma de los dados.

In [ ]:
# Clase para representar el histograma de sumas de dados
class Histograma(VGroup):
    def __init__(self, x_min=10, x_max=60,):
        super().__init__()

        # Configuracion de las caracteristicas de la escena
        self.x_min       = x_min # Valor mínimo de la suma de los dados (10 para 10 dados)
        self.x_max       = x_max # Valor máximo de la suma de los dados (60 para 10 dados)
        self.base_y      = -3.1 # Coordenada 'y' de la base del histograma
        self.ancho_total = 9.5 # Ancho total del histograma en unidades de Manim
        self.altura_bloque  = altura_objetivo / (0.06 * n_experimentos) # Altura de cada bloque del histograma, calculada para que el histograma alcance la altura objetivo
        self.ancho_bloque   = 0.16 # Ancho de cada bloque del histograma
        self.separacion     = 0.035 # Separación entre bloques del histograma
        self.color_hist     = "#7bc96f" # Color de los bloques del histograma
        self.color_activo   = "#d9a066" # Color del bloque activo 
        self.color_eje      = "#d8d8d8" # Color del eje

        # Diccionario para almacenar los bloques del histograma por cada suma de dados
        self.bloques_por_bin = defaultdict(list) 
        self.ultimo_activo   = None

        # Crear el eje horizontal del histograma
        eje = Line(
            [-self.ancho_total/2, self.base_y, 0], # Punto inicial del eje (izquierda)
            [ self.ancho_total/2, self.base_y, 0], # Punto final del eje (derecha)
            color=self.color_eje, # Color del eje
            stroke_width=1.6 # Grosor del eje
        )
        self.add(eje) # Agregar el eje al grupo del histograma

        # Crear las marcas y números en el eje horizontal
        for k in range(x_min, x_max + 1): 
            x = np.interp(
                k,
                [x_min, x_max],
                [-self.ancho_total/2, self.ancho_total/2]
            )
            # Crear una marca (tick) en el eje para cada valor de suma de dados
            tick = Line(
                [x, self.base_y - 0.07, 0],
                [x, self.base_y + 0.07, 0],
                color=self.color_eje,
                stroke_width=1
            )
            self.add(tick)
            # Agregar números al eje solo para las sumas pares para evitar saturación visual
            if k % 2 == 0:
                numero = Text(
                    str(k),
                    font_size=12,
                    color=self.color_eje
                )
                numero.move_to([x, self.base_y - 0.22, 0])
                self.add(numero) # Agregar los números al grupo del histograma

   
    # Añade un bloque al histograma para la suma dada, actualizando el bloque activo y su color
    def añadir_bloque(self, suma):
        # Cambia el color del bloque que estaba activo anteriormente al color base del histograma
        if self.ultimo_activo is not None:
            self.ultimo_activo.set_fill(self.color_hist)
        # Calcula la posición 'x' del bloque correspondiente al valor de la nueva suma
        x = np.interp(
            suma,
            [self.x_min, self.x_max],
            [-self.ancho_total/2, self.ancho_total/2]
        )
        # Calcula la posición 'y' de la columna donde debe ir colocado el nuevo bloque, teniendo en cuenta cuántos bloques ya hay apilados para esa suma
        j = len(self.bloques_por_bin[suma])
        # Calcula la posición 'y' donde debe colocarsel el nuevo bloque (base + altura de bloque/2 * número de bloques apilados)
        y = (
            self.base_y
            + self.altura_bloque / 2
            + j * self.altura_bloque
        )
        # Crea el nuevo bloque a colocar
        bloque = Rectangle(
            width=self.ancho_bloque,
            height=self.altura_bloque
        )
        bloque.set_fill(self.color_activo, opacity=1) # Color del bloque activo
        bloque.set_stroke(width=0) # Sin borde para los bloques del histograma
        bloque.move_to([x, y, 0]) # Mueve el bloque a la posición calculada

        self.bloques_por_bin[suma].append(bloque) # Guarda el bloque en el diccionario de bloques por suma
        self.ultimo_activo = bloque # Actualiza el bloque activo actual al nuevo bloque
        self.add(bloque) # Agrega el nuevo bloque al grupo del histograma

        # Devuelve el bloque creado para que pueda ser animado en la escena
        return bloque

## Distribución teórica de un dado para la clase `DistribucionDados`

Esta clase genera una representación gráfica de la distribución de probabilidad asociada al lanzamiento de un dado equilibrado de seis caras.

Cada posible resultado, desde 1 hasta 6, tiene la misma probabilidad de ocurrir. Por ello, la distribución es uniforme y todas las barras poseen la misma altura.

Cada posible resultado tiene la misma probabilidad:

$$
P(X=k)=\frac{1}{6},
\qquad k\in\{1,2,3,4,5,6\}
$$

La gráfica está formada por:

- Un eje horizontal con los valores posibles del dado.
- Una barra para cada resultado.
- Etiquetas que indican la probabilidad correspondiente.

Esta representación sirve como referencia teórica para comprender el comportamiento probabilístico de un único dado antes de analizar la distribución de las sumas obtenidas al lanzar varios dados simultáneamente.
A pesar de que esta distribución esta hecha para un dado equilibrado se podría hacer para dados trucados.

In [ ]:
# Clase para representar la distribución de probabilidades de los dados
class DistribucionDados(VGroup):
    def __init__(self, conteos):
        super().__init__()

        # Configuración de colores y dimensiones para la representación de la distribución de dados
        color_eje    = "#d8d8d8" # Color del eje
        color_barra  = "#2d9cdb" # Color de las barras 
        color_flecha = "#ffe45c" # Color de las flechas que indican la cantidad de veces que ha salido cada suma
        ancho_total  = 3.5 # Ancho total de la representación de la distribución de dados
        base_y       = 0 # Coordenada 'y' de la base de la representación de la distribución de dados
        altura_max   = 0.9 # Altura máxima de la representación de la distribución de dados (en unidades de Manim)
        probabilidad = 1/6 # Probabilidad de que salga cada suma (1/6 para cada suma de 10 dados)

        # Crear los ejes de la representación de la distribución de dados
        eje_x = Line(
            [0, base_y, 0], # Punto inicial del eje (izquierda)
            [ancho_total, base_y, 0], # Punto final del eje (derecha)
            color=color_eje, # Color del eje
            stroke_width=1.6 # Grosor del eje
        )
        eje_y = Line(
            [0, base_y, 0], # Punto inicial del eje (abajo)
            [0, altura_max, 0], # Punto final del eje (arriba)
            color=color_eje, # Color del eje
            stroke_width=1.6  # Grosor del eje
        )
        # Agregar los ejes al grupo de la distribución de dados
        self.add(eje_x, eje_y)

        # Crear las marcas y números en el eje vertical
        valores_y = [(0,    "0.00"), (0.25, "0.25"), (0.50, "0.50")] # Valores y etiquetas para las marcas en el eje vertical (probabilidades 0.00, 0.25 y 0.50)
        # Agregar las marcas y números al eje vertical
        for valor, texto in valores_y:
            y = valor 
            tick = Line(
                [-0.045, y, 0], 
                [ 0.045, y, 0],
                color=color_eje,
                stroke_width=1.2
            )
            numero = Text(
                texto,
                font_size=13,
                color=color_eje
            )
            # Colocar los números a la izquierda de las marcas del eje vertical con un pequeño espacio
            numero.next_to(tick, LEFT, buff=0.04)
            self.add(tick, numero) # Agregar los ticks y números al grupo de la distribución de dados

        # Crear las barras, dados y flechas para cada suma de dados (de 1 a 6)
        for i in range(6):
            x = 0.42 + i * 0.56 # Posición 'x' de cada barra, dado y flechas para las sumas de dados
            h = probabilidad # Altura de cada barra, proporcional a la probabilidad
           
            barra = Rectangle(width=0.48, height=h) # Crear la barra para la suma de dados correspondiente
            barra.set_fill(color_barra, opacity=0.9) # Color de las barras
            barra.set_stroke(width=0) 
            barra.move_to([x, base_y + h/2, 0]) # Mueve la barra a la posición correcta
            self.add(barra) # Agrega la barra al grupo de la distribución de dados

            dado = Dado(valor=i+1, size=0.24) # Crea un dado haciendo referencia al Vgroup `Dado` 
            dado.move_to([x, -0.30, 0]) # Mueve el dado a una posición debajo de la barra correspondiente
            self.add(dado) # Agrega el dado al grupo de la distribución de dados

            cantidad = conteos[i] # Cantidad de veces que ha salido el dado
            # Para cada vez que ha salido el dado, crear una flecha y apilarlas verticalmente con una separación de 0.07 unidades entre ellas
            for j in range(cantidad):
                flecha = Arrow(
                    start=[0, 0.06, 0], # Punto inicial de la flecha (ligeramente por encima de la barra)
                    end=[0, 0, 0], # Punto final de la flecha (en el borde superior de la barra)
                    buff=0,
                    stroke_width=2.2, # Grosor de la flecha
                    max_tip_length_to_length_ratio=0.7, # Proporción del tamaño de la punta de flecha respecto a la longitud total de la flecha
                    color=color_flecha # Color 
                )
                flecha.scale(0.11) # Escala la flecha para que tenga un tamaño adecuado para la representación
                flecha.move_to([x, h + 0.05 + j * 0.07, 0]) # Mueve la flecha a la posición correcta, apilándolas verticalmente con una separación de 0.07 unidades entre ellas
                self.add(flecha) # Agregar la flecha al grupo de la distribución de dados